# 04 · 冻结策略的离线评价

## 本页要回答什么

在 validation 已经冻结策略后，各候选策略在 test 上的 20% 预算估计值是多少？相对随机策略和 response propensity 的差值及区间如何？

## 代码导航

- 主流程阶段 8：run_pipeline.py 的 run()。
- 测试集评价入口：criteo_uplift/holdout_evaluation.py 的 evaluate_test_set()。
- 策略预算值和配对区间：criteo_uplift/evaluation.py 的 evaluate_policies()、ranking_confidence_intervals()。
- 结果文件：test_policies.csv、test_ranking.csv、curves.csv、ranking_intervals.csv、ate_test.json、predictions.npz。

这里读取冻结实验的结果，不重新训练模型，也不根据 test 结果重新选策略。通用指标推导请查 DSDAHANDBOOK；本页重点看策略结果、比较基线和不确定性边界。

In [1]:
from pathlib import Path
import sys, json, os
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'criteo_uplift').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from criteo_uplift.runtime import configure_runtime
configure_runtime()
import numpy as np
import pandas as pd
from IPython.display import display, Image, HTML
from criteo_uplift.pipeline_io import result_directory
RUN = result_directory('conversion_1000000_seed42', ROOT)
print('读取结果目录:', RUN.relative_to(ROOT))
manifest = json.loads((RUN / 'manifest.json').read_text())
print('分析对象:', manifest['primary_outcome'], '| 随机样本:', f"{manifest['sample_n']:,}")
print('seed:', manifest['seed'], '| 主预算:', manifest['selection']['primary_budget'])


读取结果目录: reports/published/conversion_1000000_seed42
分析对象: conversion | 随机样本: 1,000,000
seed: 42 | 主预算: 0.2


## 先冻结策略，再评价 test

最终策略在 validation 上选出并写入 selection.json。test 的作用是评价已经冻结的候选方案，不是再选一次赢家。即使另一个模型在 test 上点估计更高，也不能回头改选并将其当成独立验证。

“每十万人增量”以全部符合条件的待选用户为分母；20% 预算意味着每十万人选出 20,000 人。

In [ ]:
selection = json.loads((RUN / 'selection.json').read_text())
print('Validation 冻结的最终策略:', selection['selected_policy'])
policies = pd.read_csv(RUN / 'test_policies.csv')
main = policies[np.isclose(policies.budget, selection['primary_budget'])].copy()
main = main.sort_values('incremental_per_100k', ascending=False)

print('主预算下的策略点估计与 95% CI（每十万待选人）')
display(main[['policy', 'n_selected', 'incremental_per_100k', 'ci_low', 'ci_high']])

print('相对基线的成对差值与 95% CI')
display(main[[
    'policy', 'vs_random_per_100k', 'vs_random_ci_low', 'vs_random_ci_high',
    'vs_propensity_per_100k', 'vs_propensity_ci_low', 'vs_propensity_ci_high'
]])

## 成对 bootstrap：区间应怎样阅读

**实现位置：** evaluation.py 的 evaluate_policies() 与 ranking_confidence_intervals()；测试阶段由 holdout_evaluation.py 调用。

同一次按 treatment 臂分层的重抽样用于比较策略，每次按固定分数重新选 top-k。策略差值使用成对差值分布；不要用两条独立置信区间是否重叠来判断策略差异。

当前区间条件于已拟合模型、冻结分数、固定的分配概率估计和观测臂样本量；不包含重训、模型选型、分配概率估计误差或未知用户依赖。多个辅助预算比较属于探索性分析。

In [ ]:
display(Image(filename=str(RUN / 'figures/policy.png')))

# 图展示多个预算；这里仅列主预算结果，避免再输出一张过宽的全预算表。
main_budget = selection['primary_budget']
main_budget_table = policies[np.isclose(policies.budget, main_budget)].copy()
main_budget_table = main_budget_table.sort_values('incremental_per_100k', ascending=False)
display(main_budget_table[['policy', 'budget', 'incremental_per_100k', 'ci_low', 'ci_high']])

## 本轮结论如何表达

1. 先看冻结策略在 20% 主预算下的增量估计和相对基线的成对区间。
2. 如果相对 response 的差值区间包含 0，应写“证据不足”，不要写“没有差异”，也不要强行宣称 uplift 模型胜出。
3. conversion 事件稀疏、control 臂更少，复杂模型可能不稳定；100 万行开发实验不等同于全量建模结论。
4. 这里是公开样本上的离线策略模拟，不是实际广告投放或真实收入增长。
5. 开发期稳定性、SQL 核对和成本情景见 Notebook 06；这些分析不能替代独立线上 A/B。

结果文件字段和可对外表达边界见 ../docs/RESULTS_GUIDE.md。

In [4]:
row=main[main.policy==selection['selected_policy']].iloc[0]
print(f"冻结政策 {row.policy}：每十万待选用户增量 {row.incremental_per_100k:.1f}，95% CI [{row.ci_low:.1f}, {row.ci_high:.1f}]")
print(f"相对 random 差值 {row.vs_random_per_100k:.1f}，成对95% CI [{row.vs_random_ci_low:.1f}, {row.vs_random_ci_high:.1f}]")
print('完整报告:', (RUN/'report.html').relative_to(ROOT))

冻结政策 propensity：每十万待选用户增量 117.2，95% CI [64.1, 166.2]
相对 random 差值 92.6，成对95% CI [50.0, 132.2]
完整报告: reports/published/conversion_1000000_seed42/report.html


## 下一步阅读

- 想知道 test 指标怎样计算：criteo_uplift/evaluation.py。
- 想知道测试集评价如何调用指标并写出 CSV/NPZ：criteo_uplift/holdout_evaluation.py。
- 想知道报告图表如何生成：criteo_uplift/reporting.py。

主流程与结果目录导航见 ../docs/PROJECT_PIPELINE.md；各结果文件含义见 ../docs/RESULTS_GUIDE.md。